In [2]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch.distributions import Independent, Uniform, Normal, MultivariateNormal
from sbi import analysis, utils
from sbi.inference import NPE, simulate_for_sbi
from sbi.utils.user_input_checks import (
    check_sbi_inputs,
    process_prior,
    process_simulator,
)
import math
import sympy as sp

seed = 0
torch.manual_seed(seed);


C = 299792.458 # km/s

In [6]:
def calc_hubble_distance(hubble, matter, curv, z):
    matter_term = matter * (1 + z)**3
    curvature_term = curv * (1 + z)**2
    lambda_term = 1 - matter - curv
    hubble_term = hubble * (1 + z)
    return torch.max(hubble * torch.sqrt(matter_term + curvature_term + lambda_term), torch.tensor(0.0001))

def calc_comoving_distance(hubble, matter, curv, z):
    hubble_distance = calc_hubble_distance(hubble, matter, curv, z)
    z_values = torch.linspace(0, z, 1000)
    hubble_distance_values = calc_hubble_distance(hubble, matter, curv, z_values)
    return C * torch.trapz(1 / hubble_distance_values, z_values)

def calc_luminosity_distance(comoving_distance, Ok, z):
    # Handle the flat universe case first
    if abs(Ok) < 1e-10:
        return (1 + z) * comoving_distance
    # For curved universes, use the exact formula
    Ok_abs = abs(Ok)
    sqrt_Ok = torch.sqrt(Ok_abs)
    if Ok > 0:
        # Open universe
        sinh_term = torch.sinh(sqrt_Ok * comoving_distance / C)
        transverse_distance = C / sqrt_Ok * sinh_term
    else:
        # Closed universe
        sin_term = torch.sin(sqrt_Ok * comoving_distance / C)
        transverse_distance = C / sqrt_Ok * sin_term
    return (1 + z) * transverse_distance

def calc_apparent_magnitude(hubble, matter, curv, z):
    luminosity_distance = calc_luminosity_distance(hubble, matter, curv, z)
    return 5 * torch.log10(luminosity_distance) + 25

def load_real_data():
    """Load and prepare the supernova data"""
    import pandas as pd
    df = pd.read_csv('/Users/yhra/Documents/Master/Semester_3/BATIP/Supernova_project/Data/Pantheon+SH0ES.dat', sep='\s+')
    
    # Filter data
    mask = df['zCMB'] > 0.001
    z_obs = df['zCMB'][mask].values
    mu_obs = df['MU_SH0ES'][mask].values
    mu_err = df['MU_SH0ES_ERR_DIAG'][mask].values
    
    # Convert to tensors
    z_obs = torch.as_tensor(z_obs, dtype=torch.float32).clone().detach()
    mu_obs = torch.as_tensor(mu_obs, dtype=torch.float32).clone().detach()
    mu_err = torch.as_tensor(mu_err, dtype=torch.float32).clone().detach()

    # Create covariance matrix
    cov_data = np.loadtxt('/Users/yhra/Documents/Master/Semester_3/BATIP/Supernova_project/Data/Pantheon+SH0ES_STATONLY.cov')
    cov_matrix = _create_cov_mat(cov_data)
    
    return z_obs, mu_obs, mu_err, cov_matrix

def _create_cov_mat(cov_data):
    n = int(cov_data[0])
    cov_matrix = cov_data[1:].reshape(n, n)
    
    # Make symmetric if needed
    if not np.allclose(cov_matrix, cov_matrix.T):
        cov_matrix = (cov_matrix + cov_matrix.T) / 2
    return cov_matrix

# test on fixed values
Ho = torch.tensor(70.0)
Om = torch.tensor(0.3)
z = torch.tensor(0.1)
Ok = torch.tensor(0.01)
comoving_distance = calc_comoving_distance(Ho, Om, Ok, z)
luminosity_distance = calc_luminosity_distance(comoving_distance, Ok, z)
print(luminosity_distance)
print(comoving_distance)



tensor(460.0784)
tensor(418.2531)


In [39]:
import sympy as sp

def taylor_series_sinh_sqrt(x_order=16):
    # Define symbols
    x, a = sp.symbols('x a')
    
    # Define the function
    f = (1 / sp.sqrt(x)) * sp.sinh(sp.sqrt(a) * x)
    
    # Compute the Taylor series expansion
    taylor_series = sp.series(f, x, 0, x_order + 1).removeO()
    
    return taylor_series

# Example usage
series = taylor_series_sinh_sqrt(16)
print(series)

# evaluate the series at x = 0.1 and a = 0.1
series.subs({'x': 0.1, 'a': 0.1}).evalf()



a**(17/2)*x**(33/2)/355687428096000 + a**(15/2)*x**(29/2)/1307674368000 + a**(13/2)*x**(25/2)/6227020800 + a**(11/2)*x**(21/2)/39916800 + a**(9/2)*x**(17/2)/362880 + a**(7/2)*x**(13/2)/5040 + a**(5/2)*x**(9/2)/120 + a**(3/2)*x**(5/2)/6 + sqrt(a)*sqrt(x)


0.100016667500020